# Plan A (Colab): Emergency Multimodal Pipeline

Streams CheXpert from Hugging Face, creates the high-risk label, extracts embeddings, writes imaging artifacts, and runs multimodal fusion with age + sex.

In [ ]:
!pip -q install datasets xgboost scikit-learn pandas numpy matplotlib pillow torch torchvision

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from PIL import Image
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
import matplotlib.pyplot as plt
import torchvision.models as models

RANDOM_STATE = 42
N_TARGET = 2000
HF_DATASET = "StanfordAIMI/CheXpert-v1.0-512"  # fallback: danjacobellis/chexpert
HF_SPLIT = "train"
SEVERE_LABELS = ["Pneumonia", "Edema", "Consolidation", "Pleural Effusion"]
CSV_PATH = "/content/drive/MyDrive/BREATHE/midpoint/df_chexpert_plus_240401.csv"
OUT_DIR = Path("/content/drive/MyDrive/BREATHE_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

In [ ]:
def pick_path(ex):
    for k in ["path_to_image", "Path", "path", "image_path", "filepath"]:
        if k in ex and ex[k] is not None:
            return str(ex[k])
    return None

def get_label_val(ex, key):
    candidates = [key, key.lower(), key.replace(" ", "_"), key.replace(" ", "")]
    for c in candidates:
        if c in ex:
            return ex[c]
    return None

def high_risk_label(ex):
    vals = []
    for k in SEVERE_LABELS:
        v = get_label_val(ex, k)
        if v is None:
            return None
        if isinstance(v, (list, tuple)) and len(v) > 0:
            v = v[0]
        try:
            v = float(v)
        except Exception:
            return None
        if v == -1:
            return None
        vals.append(v)
    return int(sum(vals) > 0)

def get_image(ex):
    for k in ["image", "Image", "img"]:
        if k in ex and ex[k] is not None:
            return ex[k]
    return None

stream = load_dataset(HF_DATASET, split=HF_SPLIT, streaming=True)
rows = []
for ex in stream:
    p = pick_path(ex)
    y = high_risk_label(ex)
    im = get_image(ex)
    if p is None or y is None or im is None:
        continue
    rows.append({"path_to_image": p, "Severe": y, "image": im})
    if len(rows) >= N_TARGET:
        break

hf_df = pd.DataFrame(rows)
print("HF rows collected:", len(hf_df))
print(hf_df["Severe"].value_counts(dropna=False))

In [ ]:
demo = pd.read_csv(CSV_PATH, usecols=["path_to_image", "age", "sex"])
demo["age"] = pd.to_numeric(demo["age"], errors="coerce")
demo["sex"] = demo["sex"].astype(str).str.strip().str.lower().map({"male": 1, "female": 0})
demo = demo.dropna(subset=["age", "sex"]).copy()
demo["sex"] = demo["sex"].astype(int)

merged = hf_df.merge(demo, on="path_to_image", how="inner")
print("Merged rows (HF image + demographics):", len(merged))
print(merged["Severe"].value_counts(dropna=False))
assert len(merged) > 200, "Too few merged rows; check dataset path field compatibility."

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
embedding_dim = model.fc.in_features
model.fc = nn.Identity()
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
preprocess = weights.transforms()

embs = []
paths = []
labels = []

with torch.no_grad():
    for _, r in merged.iterrows():
        img = r["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        x = preprocess(img.convert("RGB")).unsqueeze(0).to(device)
        e = model(x).detach().cpu().numpy().reshape(-1)
        embs.append(e)
        paths.append(r["path_to_image"])
        labels.append(int(r["Severe"]))

emb_array = np.stack(embs, axis=0)
np.savez_compressed(
    OUT_DIR / "imaging_embeddings_resnet18.npz",
    path_to_image=np.asarray(paths),
    embeddings=emb_array,
)
print("Saved:", OUT_DIR / "imaging_embeddings_resnet18.npz", emb_array.shape)

In [ ]:
X_img = pd.DataFrame(emb_array, columns=[f"emb_{i}" for i in range(emb_array.shape[1])])
y = pd.Series(labels)
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_validate(clf, X_img, y, cv=cv, scoring={"acc": "accuracy", "f1": "f1"}, n_jobs=-1)
X_tr, X_te, y_tr, y_te = train_test_split(X_img, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
clf.fit(X_tr, y_tr)
pred = clf.predict(X_te)
prob = clf.predict_proba(X_te)[:, 1]

imaging_metrics = {
    "label_definition": "High-Risk=any(Pneumonia,Edema,Consolidation,Pleural Effusion)",
    "n_samples": int(len(y)),
    "cv_accuracy_mean": float(np.mean(scores["test_acc"])),
    "cv_accuracy_std": float(np.std(scores["test_acc"])),
    "cv_f1_mean": float(np.mean(scores["test_f1"])),
    "cv_f1_std": float(np.std(scores["test_f1"])),
    "holdout_accuracy": float(accuracy_score(y_te, pred)),
    "holdout_f1": float(f1_score(y_te, pred)),
    "holdout_precision": float(precision_score(y_te, pred, zero_division=0)),
    "holdout_recall": float(recall_score(y_te, pred, zero_division=0)),
    "holdout_roc_auc": float(roc_auc_score(y_te, prob)),
    "holdout_confusion_matrix": confusion_matrix(y_te, pred).tolist(),
}
with open(OUT_DIR / "imaging_metrics.json", "w") as f:
    json.dump(imaging_metrics, f, indent=2)
print("Saved:", OUT_DIR / "imaging_metrics.json")

In [ ]:
full = merged[["path_to_image", "age", "sex", "Severe"]].copy()
emb_df = pd.DataFrame(emb_array, columns=[f"emb_{i}" for i in range(emb_array.shape[1])])
X_multi = pd.concat([emb_df, full[["age", "sex"]].reset_index(drop=True)], axis=1)
y_multi = full["Severe"].reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
spaces = {
    "logistic_regression": (
        Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE))]),
        {"clf__C": [0.01, 0.1, 1.0, 10.0]},
    ),
    "gradient_boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {"n_estimators": [100, 200], "max_depth": [3, 5], "learning_rate": [0.01, 0.1]},
    ),
    "mlp": (
        Pipeline([("scaler", StandardScaler()), ("clf", MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE))]),
        {"clf__hidden_layer_sizes": [(128,), (256, 64)], "clf__alpha": [1e-4, 1e-3]},
    ),
}
try:
    from xgboost import XGBClassifier
    spaces["xgboost"] = (
        XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0, n_jobs=-1),
        {"n_estimators": [100, 200], "max_depth": [3, 5], "learning_rate": [0.01, 0.1]},
    )
except Exception:
    pass

mm = {"models": {}}
best_name = None
best_est = None
best_f1 = -1
for name, (est, grid) in spaces.items():
    gs = GridSearchCV(est, grid, cv=skf, scoring="f1", refit=True, n_jobs=-1)
    gs.fit(X_multi, y_multi)
    cv_scores = cross_validate(gs.best_estimator_, X_multi, y_multi, cv=skf, scoring={"acc":"accuracy","f1":"f1"}, n_jobs=-1)
    mm["models"][name] = {
        "best_params": gs.best_params_,
        "cv_accuracy_mean": float(np.mean(cv_scores["test_acc"])),
        "cv_accuracy_std": float(np.std(cv_scores["test_acc"])),
        "cv_f1_mean": float(np.mean(cv_scores["test_f1"])),
        "cv_f1_std": float(np.std(cv_scores["test_f1"])),
    }
    if mm["models"][name]["cv_f1_mean"] > best_f1:
        best_f1 = mm["models"][name]["cv_f1_mean"]
        best_name = name
        best_est = gs.best_estimator_

X_tr, X_te, y_tr, y_te = train_test_split(X_multi, y_multi, test_size=0.2, stratify=y_multi, random_state=RANDOM_STATE)
best_est.fit(X_tr, y_tr)
pred = best_est.predict(X_te)
out = {
    "label_definition": "High-Risk=any(Pneumonia,Edema,Consolidation,Pleural Effusion)",
    "n_samples": int(len(y_multi)),
    "best_model_by_cv_f1": best_name,
    "models": mm["models"],
    "holdout": {
        "accuracy": float(accuracy_score(y_te, pred)),
        "f1": float(f1_score(y_te, pred)),
        "precision": float(precision_score(y_te, pred, zero_division=0)),
        "recall": float(recall_score(y_te, pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_te, pred).tolist(),
    },
}
if hasattr(best_est, "predict_proba"):
    p = best_est.predict_proba(X_te)[:, 1]
    out["holdout"]["roc_auc"] = float(roc_auc_score(y_te, p))

with open(OUT_DIR / "multimodal_metrics.json", "w") as f:
    json.dump(out, f, indent=2)
print("Saved:", OUT_DIR / "multimodal_metrics.json")
print("Best multimodal model:", best_name)
print("Holdout F1:", out["holdout"]["f1"])

In [ ]:
# Final figures: multimodal confusion matrix + unimodal vs multimodal bar chart
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

with open(OUT_DIR / "imaging_metrics.json", "r") as f:
    imaging = json.load(f)
with open(OUT_DIR / "multimodal_metrics.json", "r") as f:
    multi = json.load(f)

# 1) Multimodal confusion matrix figure
cm = np.array(multi["holdout"]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(4.2, 4.2))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, cmap="Purples", colorbar=False)
ax.set_title(f"Best multimodal: {multi['best_model_by_cv_f1']}")
plt.tight_layout()
cm_path = OUT_DIR / "multimodal_confusion_matrix_best.png"
fig.savefig(cm_path, dpi=150)
plt.close(fig)

# 2) Tabular vs Imaging vs Multimodal accuracy/F1 bar chart
tab_acc = float("nan")
tab_f1 = float("nan")
tab_path = OUT_DIR / "tabular_metrics.json"
if tab_path.exists():
    with open(tab_path, "r") as f:
        tab = json.load(f)
    if "models" in tab:
        best_k = max(
            [k for k, v in tab["models"].items() if isinstance(v, dict) and "cv_f1_mean" in v],
            key=lambda k: tab["models"][k]["cv_f1_mean"],
        )
        tab_acc = tab["models"][best_k]["cv_accuracy_mean"]
        tab_f1 = tab["models"][best_k]["cv_f1_mean"]

img_acc = imaging.get("cv_accuracy_mean", float("nan"))
img_f1 = imaging.get("cv_f1_mean", float("nan"))
mm_acc = multi["holdout"]["accuracy"]
mm_f1 = multi["holdout"]["f1"]

labels = ["Tabular", "Imaging", "Multimodal"]
acc = [tab_acc, img_acc, mm_acc]
f1 = [tab_f1, img_f1, mm_f1]
x = np.arange(len(labels))
w = 0.34
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.bar(x - w / 2, acc, w, label="Accuracy")
ax.bar(x + w / 2, f1, w, label="F1")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_title("Tabular vs Imaging vs Multimodal")
ax.legend()
plt.tight_layout()
cmp_path = OUT_DIR / "multimodal_vs_unimodal_accuracy_f1.png"
fig.savefig(cmp_path, dpi=150)
plt.close(fig)

print("Saved:", cm_path)
print("Saved:", cmp_path)
print("Note: if tabular_metrics.json is absent, Tabular bar will appear as NaN.")